# Karhutla: Progres Penanganan Lahan per Provinsi (laporan 16 Oktober 2026)

**Pertanyaan:** Provinsi mana yang paling tertinggal dalam pemadaman lahan, dan bagaimana kapasitas penanganannya?
**Sumber data:** BNPB, laporan per 16 Oktober 2026. Sebagian narasi di laporan merujuk 14-15 Oktober.
**Cakupan:** 6 provinsi, satu laporan, jadi ini potret satu hari, bukan tren.

In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt

In [ ]:
PATH = '/kaggle/input/datasets/zikriazzuri/karhutla/karhutla-2026_semua-provinsi_2026-10-16.csv'
df_raw = pd.read_csv(PATH)
df = df_raw.copy()

print(df.shape)
df.info()

In [ ]:
df.columns = (df.columns.str.strip().str.lower()
              .str.replace(r'[^\w]+', '_', regex=True).str.strip('_'))
df['tanggal_laporan'] = pd.to_datetime('2026-10-16')

df.columns.tolist()

In [ ]:
df['provinsi'] = df['provinsi'].str.strip().replace({
    'Sumsel': 'Sumatera Selatan', 'Kalbar': 'Kalimantan Barat',
    'Kalteng': 'Kalimantan Tengah', 'Kalsel': 'Kalimantan Selatan'})

In [ ]:
df.isna().sum()[lambda s: s > 0]

In [ ]:
cek = df[['hotspot_rendah','hotspot_sedang','hotspot_tinggi']].fillna(0).sum(axis=1)
assert (cek == df['total_hotspot']).all(), 'Total hotspot tidak cocok'
print('Total hotspot cocok di semua provinsi')
df['hotspot_rendah'] = df['hotspot_rendah'].fillna(0)

In [ ]:
kolom_angka = ['personil_pemerintah_pusat','relawan','polisi_hutan','mpa']
for c in kolom_angka:
    print(c, df[c].unique())
print('jumlah_wb_harian (teks campuran):', df['jumlah_wb_harian'].unique())

In [ ]:
for c in kolom_angka:
    df[c] = pd.to_numeric(
        df[c].astype(str).str.replace(',', '.').replace({'-': np.nan, 'nan': np.nan}),
        errors='coerce')
df[kolom_angka]

In [ ]:
wb = df_raw['jumlah WB harian'].astype(str).str.lower().str.strip()
df['wb_drop_harian'] = pd.to_numeric(wb.str.extract(r'(\d+)\s*(?:kali|x)')[0], errors='coerce')
df['wb_liter_teks'] = pd.to_numeric(
    wb.str.extract(r'([\d\.]+)\s*l')[0].str.replace('.', '', regex=False), errors='coerce')
df.loc[wb == '0', ['wb_drop_harian', 'wb_liter_teks']] = 0

assert (df['wb_liter_teks'] == df['volume_wb_ltr']).all(), 'Volume WB dari teks tidak sama dengan volume_wb_ltr'
print('Volume WB dari teks sama dengan volume_wb_ltr')
df[['provinsi', 'jumlah_wb_harian', 'wb_drop_harian', 'wb_liter_teks', 'volume_wb_ltr']]

In [ ]:
df = df.drop(columns='jumlah_wb_harian')
df = df.drop(columns='wb_liter_teks')

In [ ]:

s = (df['kualitas_udara_pm_25'].str.lower()
       .str.replace('sanagat', 'sangat').str.strip())
parts = s.str.split('-', expand=True).apply(lambda c: c.str.strip())
skala = {'baik':1, 'sedang':2, 'tidak sehat':3, 'sangat tidak sehat':4, 'berbahaya':5}
df['udara_terbaik']  = parts[0].map(skala)
df['udara_terburuk'] = parts[1].fillna(parts[0]).map(skala)
df[['provinsi','kualitas_udara_pm_25','udara_terbaik','udara_terburuk']]

In [ ]:
nums = (df['jarak_pandang_bandara_km'].astype(str).str.replace(',', '.')
          .str.findall(r'\d+\.?\d*'))
df['jarak_pandang_min'] = nums.apply(lambda x: min(map(float, x)) if x else np.nan)
df['jarak_pandang_max'] = nums.apply(lambda x: max(map(float, x)) if x else np.nan)
df['jarak_pandang_batas'] = df['jarak_pandang_bandara_km'].astype(str).str.contains('[<≤]')
df[['provinsi','jarak_pandang_bandara_km','jarak_pandang_min','jarak_pandang_max','jarak_pandang_batas']]

In [ ]:
df.loc[df['provinsi'] == 'Kalimantan Selatan', 'jarak_pandang_min'] = np.nan
df[['provinsi','jarak_pandang_bandara_km','jarak_pandang_min','jarak_pandang_max','jarak_pandang_batas']]

In [ ]:
komp_personel = ['personil_bpbd','personil_pemerintah_pusat','tni','polri','pemda',
                 'relawan','polisi_hutan','manggala_agni','mpa','satpol_pp','perusahaan']
df['selisih_personel'] = df['total_personel'] - df[komp_personel].fillna(0).sum(axis=1)
df['selisih_standby'] = df['total_unit_standby'] - df[['unit_patroli_standby','unit_wb_standby','unit_omc_standby']].sum(axis=1)

assert (df['selisih_personel'].abs() < 1e-9).all() and (df['selisih_standby'] == 0).all(), 'Total tidak cocok dengan komponennya'
print('Total personel dan unit standby cocok dengan komponennya di semua provinsi')
df[['provinsi','selisih_personel','selisih_standby']]

In [ ]:
komp_personel = ['personil_bpbd','personil_pemerintah_pusat','tni','polri','pemda',
                 'relawan','polisi_hutan','manggala_agni','mpa','satpol_pp','perusahaan']
df[komp_personel] = df[komp_personel].fillna(0)

kolom_inti = ['provinsi','tanggal_laporan','hotspot_rendah','hotspot_sedang','hotspot_tinggi',
              'total_hotspot','fire_spot','lahan_terbakar_harian_ha','udara_terburuk',
              'jarak_pandang_min','kabkot_terdampak','total_personel','patroli_unit',
              'wb_unit','wb_drop_harian','volume_wb_ltr','omc_unit',
              'luas_lahan_ditangani_darat','luas_lahan_ditangani_udara',
              'total_unit_udara_hari_ini','total_unit_standby']
df_clean = df[kolom_inti].rename(columns={'lahan_terbakar_harian_ha': 'lahan_harus_dipadamkan_ha'})
df_clean.info()
print(df_clean.isna().sum())
df_clean.to_csv('karhutla_clean.csv', index=False)

In [ ]:
m = df_clean.copy()
m['pct_hotspot_tinggi']         = m['hotspot_tinggi'] / m['total_hotspot'] * 100
m['personel_per_hotspot']       = m['total_personel'] / m['total_hotspot']
m['unit_udara_per_100_hotspot'] = m['total_unit_udara_hari_ini'] / m['total_hotspot'] * 100
m['rank_hotspot']               = m['total_hotspot'].rank(ascending=False).astype(int)
m['rank_personel_per_hotspot']  = m['personel_per_hotspot'].rank(ascending=False).astype(int)

m[['provinsi','total_hotspot','total_personel','pct_hotspot_tinggi',
   'personel_per_hotspot','unit_udara_per_100_hotspot',
   'rank_hotspot','rank_personel_per_hotspot']].round(2).sort_values('rank_hotspot')

In [ ]:
kolom_p = ['personil BPBD','personil pemerintah pusat','TNI','Polri','Pemda',
           'Relawan','Polisi Hutan','Manggala Agni','MPA','Satpol PP','Perusahaan','Total Personel']
df_raw.loc[df_raw['Provinsi'] == 'Kalteng', kolom_p].T


In [ ]:
m['personel_per_fire_spot'] = m['total_personel'] / m['fire_spot']
m['personel_per_ha']        = m['total_personel'] / m['lahan_harus_dipadamkan_ha']
m['pct_hotspot_jadi_api']   = m['fire_spot'] / m['total_hotspot'] * 100

m.to_csv('karhutla_metrik.csv', index=False)
m[['provinsi','personel_per_fire_spot','personel_per_ha','pct_hotspot_jadi_api']].round(2)

In [ ]:
kendala = pd.DataFrame([
    ['Riau', 'Meminta dukungan dana operasional (uang semangat) untuk menjaga moril satgas', 'dana',
     'Rekomendasi: satgas baru didukung untuk 10 hari', 'tertulis'],
    ['Jambi', 'Tidak ada kendala atau permintaan yang disebut', None, 'Rekomendasi 1-5', 'tidak disebut eksplisit'],
    ['Sumatera Selatan', 'Operasi udara belum optimal karena keterbatasan jarak pandang', 'cuaca/visibilitas',
     'Kesimpulan', 'tertulis'],
    ['Sumatera Selatan', 'OMC dihentikan karena kondisi atmosfer tidak mendukung', 'cuaca/visibilitas',
     'Kesimpulan', 'tertulis'],
    ['Kalimantan Barat', 'Helikopter VN-8424 berstatus Un-Serviceable', 'armada', 'Rekomendasi', 'tertulis'],
    ['Kalimantan Tengah', 'Tidak ada kendala yang disebut; operasi udara tidak dilaksanakan, alasan tidak disebut',
     None, 'Kesimpulan b', 'tidak disebut eksplisit'],
    ['Kalimantan Selatan', 'Meminta kendaraan 4WD dan motor pemadam', 'peralatan darat', 'Rekomendasi 1', 'tertulis'],
    ['Kalimantan Selatan', 'Meminta tambahan helikopter water bombing', 'armada', 'Rekomendasi 2', 'tertulis'],
    ['Kalimantan Selatan', 'Meminta APD 750 personel, uang lelah, dan BBM', 'dana', 'Rekomendasi 3', 'tertulis'],
    ['Kalimantan Selatan', 'Akses dan sumber air terbatas; kemarau dan lahan gambut menyulitkan pemadaman',
     'medan/akses', 'Kesimpulan', 'tertulis'],
], columns=['provinsi','kendala','kategori','dasar','jenis'])

kendala.to_csv('karhutla_kendala.csv', index=False)
kendala['jenis'].value_counts()

In [ ]:
p = df[['provinsi','lahan_terbakar_harian_ha','luas_lahan_yang_akan_dipadamkan',
        'luas_lahan_ditangani_darat','luas_lahan_ditangani_udara']].copy()

p['dipadamkan_ha']  = p['luas_lahan_ditangani_darat'] + p['luas_lahan_ditangani_udara']
p['sisa_hitung_ha'] = p['lahan_terbakar_harian_ha'] - p['dipadamkan_ha']
p['selisih_sisa']   = p['sisa_hitung_ha'] - p['luas_lahan_yang_akan_dipadamkan']
p['pct_tertangani'] = (p['dipadamkan_ha'] / p['lahan_terbakar_harian_ha'] * 100).round(1)
p['pct_udara']      = (p['luas_lahan_ditangani_udara'] / p['dipadamkan_ha'] * 100).round(1)
p

In [ ]:

df_raw[['Provinsi','Lahan Terbakar Harian (Ha)','luas lahan yang akan dipadamkan',
        'luas lahan ditangani darat','luas lahan ditangani udara']]

In [ ]:
p['sisa_dilaporkan_ha'] = p['luas_lahan_yang_akan_dipadamkan']
p['sisa_ha'] = p['sisa_hitung_ha']                       # dipakai untuk analisis
p['flag_data'] = np.where(p['selisih_sisa'].abs() > 0.01,
                          'sisa dilaporkan tidak cocok; dipakai hasil hitung', '')

total_sisa = p['sisa_ha'].sum()
p['pct_dari_total_sisa'] = (p['sisa_ha'] / total_sisa * 100).round(1)

p[['provinsi','lahan_terbakar_harian_ha','luas_lahan_ditangani_darat','luas_lahan_ditangani_udara',
   'dipadamkan_ha','sisa_ha','pct_tertangani','pct_udara','pct_dari_total_sisa','flag_data']]

In [ ]:
final = p.rename(columns={'lahan_terbakar_harian_ha': 'lahan_harus_dipadamkan_ha'})
final = final[['provinsi','lahan_harus_dipadamkan_ha','luas_lahan_ditangani_darat',
               'luas_lahan_ditangani_udara','dipadamkan_ha','sisa_dilaporkan_ha','sisa_ha',
               'pct_tertangani','pct_udara','pct_dari_total_sisa','flag_data']]
final = final.sort_values('pct_tertangani').reset_index(drop=True)
final.to_csv('karhutla_progres_penanganan.csv', index=False)
final

In [ ]:
o = final.sort_values('sisa_ha', ascending=False)
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(o['provinsi'], o['luas_lahan_ditangani_darat'], label='Dipadamkan darat')
ax.bar(o['provinsi'], o['luas_lahan_ditangani_udara'],
       bottom=o['luas_lahan_ditangani_darat'], label='Dipadamkan udara')
ax.bar(o['provinsi'], o['sisa_ha'], bottom=o['dipadamkan_ha'], label='Sisa')
ax.set_ylabel('Hektare')
ax.set_title('Progres penanganan lahan per provinsi (16 Okt 2026)')
ax.legend()
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

In [ ]:
o = final[final['provinsi'] != 'Riau'].sort_values('pct_tertangani')
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(o['provinsi'], o['pct_tertangani'])
for i, v in enumerate(o['pct_tertangani']):
    ax.text(v + 1, i, f'{v:.1f}%', va='center')
ax.set_xlim(0, 105); ax.set_xlabel('% lahan yang sudah dipadamkan')
ax.set_title('Persentase tertangani per provinsi (Riau tidak ditampilkan: lahan hanya 2,5 ha)')
plt.tight_layout(); plt.show()

## Temuan
- Kalimantan Tengah punya persentase tertangani terendah (12,4%) dan menyumbang 76% dari 685,52 ha lahan yang belum padam; operasi udara tidak dilaksanakan.
- Jambi kedua terendah (25,2%).
- Laporan tidak menyebut penyebabnya.

## Batasan
- Satu laporan (16 Okt 2026), enam provinsi: potret, bukan tren.
- Persentase dihitung terhadap seluruh lahan yang harus dipadamkan, termasuk sisa hari sebelumnya.
- Kalsel: sisa dilaporkan 614,00 ha tidak konsisten; dipakai hasil hitung 6,60 ha.
- Narasi sebagian provinsi merujuk 14-15 Oktober.